# Workflow Test (Stage 13)

This notebook runs the full **LangGraph agent workflow** on retrieved context:

`SummaryAgent → ConceptAgent → QuizAgent → FlashcardAgent → MindMapAgent`

**Prerequisites:**
- Index already created (`01_test_rag_pipeline.ipynb`, Part A) **or** run Part A below
- `.env` with `HUGGINGFACE_API_TOKEN` (Stage 7)
- `uv sync`

## Configuration

In [ ]:
import json
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VECTOR_DB_PATH = ROOT / "data/vector_db"

TOP_K = 5
QUESTION = "What are the main contributions of this paper?"

token = os.getenv("HUGGINGFACE_API_TOKEN") or os.getenv("HF_TOKEN")

print(f"Project root: {ROOT}")
print(f"Vector DB: {VECTOR_DB_PATH}")
print(f"Question: {QUESTION}")
print(f"HF token: {'configured' if token else 'NOT configured'}")

## 1. Load retriever

In [ ]:
from rag.embeddings import ChunkEmbedder
from rag.retriever import DocumentRetriever
from rag.vectorstore import ChromaVectorStore

store = ChromaVectorStore(persist_directory=VECTOR_DB_PATH)
store.load_collection()

chunk_count = store.collection.count()
print(f"Collection: {store.collection_name} ({chunk_count} chunks)")

if chunk_count == 0:
    raise RuntimeError("Empty index. Run notebook 01_test_rag_pipeline (Part A) first.")

embedder = ChunkEmbedder()
retriever = DocumentRetriever(vector_store=store, embedder=embedder, top_k=TOP_K)

## 2. Retrieve context

In [ ]:
context = retriever.invoke(QUESTION)
print(f"Chunks retrieved: {len(context)}")

## 3. Run LangGraph workflow

In [ ]:
from graph.workflow import run_workflow

print("Running full agent workflow (this may take a few minutes)...")
result = run_workflow(QUESTION, context)

## 4. Inspect outputs

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(result["summary"]["executive_summary"])
print()
for insight in result["summary"]["key_insights"]:
    print(f"- {insight}")

In [ ]:
print("=" * 60)
print("CONCEPTS")
print("=" * 60)
for item in result["concepts"]["concepts"]:
    print(f"\n{item['concept']}")
    print(f"  Definition: {item['definition']}")
    print(f"  Relevance:  {item['relevance']}")

In [ ]:
print("=" * 60)
print("QUIZ")
print("=" * 60)
for item in result["quiz"]["questions"]:
    print(f"\n[{item['difficulty'].upper()}] {item['question']}")
    print(f"Answer: {item['answer']}")

In [ ]:
print("=" * 60)
print("FLASHCARDS")
print("=" * 60)
for card in result["flashcards"]["flashcards"]:
    print(f"\nFRONT: {card['front']}")
    print(f"BACK:  {card['back']}")

In [ ]:
print("=" * 60)
print("MIND MAP")
print("=" * 60)
print(result["mindmap"]["text"])

In [ ]:
print("Full workflow state (JSON):")
print(json.dumps({
    "question": QUESTION,
    "summary": result["summary"],
    "concepts": result["concepts"],
    "quiz": result["quiz"],
    "flashcards": result["flashcards"],
    "mindmap": result["mindmap"],
}, indent=2, ensure_ascii=False))